In [0]:
df1 = spark.read.format("csv").option("header", "true").load("dbfs:/FileStore/shared_uploads/50030877@serc.ac.uk/ufo_truth-1.csv")
df1.show()

In [0]:
df2 = spark.read.format("csv").option("header", "true").load("dbfs:/FileStore/shared_uploads/50030877@serc.ac.uk/UFO_Sightings-1.csv")
df2.show()

In [0]:
df_truth = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("dbfs:/FileStore/shared_uploads/50030877@serc.ac.uk/ufo_truth.csv")
)
 
display(df_truth)

In [0]:
df_truth.printSchema()

In [0]:
df_sightings = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("dbfs:/FileStore/shared_uploads/50030877@serc.ac.uk/UFO_Sightings.csv")
)
 
display(df_sightings)

In [0]:
df_sightings.printSchema()

In [0]:
df_sightings.show(10)

## Select some useful columns 
Here we createasmaller DataFrame using the column names from the CSV File

In [0]:
ufo_small = df_sightings.select("city", "country", "shape")  

display(ufo_small) 

## Count sightings by shape

In [0]:
shape_counts = (
    df_sightings
    .groupBy("shape")
    .count()
    .orderBy("count", ascending=False)
)

display(shape_counts)

# Count Sightings by country

In [0]:
country_counts = (
    df_sightings
    .groupBy("country")
    .count()
    .orderBy("count", ascending=False)
)

display(country_counts)

## Filter sightings from the US

In [0]:
us_sightings = df_sightings.filter(df_sightings.country == "us")

display(us_sightings)

## Select columns with spaces in their names

Some colums contain spaces and brackets, we must use their exact name otherwise things will not work.

In [0]:
durations = df_sightings.select("city", "shape","duration (seconds)", "duration (hours/min)")

display(durations)

## Clean the Column Name

This makes the dataset easier to work with by renaming columns that contain spaces or trailing spaces.

In [0]:
df_sightings_clean = (
    df_sightings
    .withColumnRenamed("duration (seconds)", "duration_seconds")
    .withColumnRenamed("duration (hours/min)", "duration_hours_min")
    .withColumnRenamed("date posted", "date_posted")
    .withColumnRenamed("longitude ", "longitude")
)
 
display(df_sightings_clean)

## check the cleaned schema 

In [0]:
df_sightings_clean.printSchema()

## Count sightings by state

In [0]:
state_counts = (
    df_sightings_clean
    .groupBy("state")
    .count()
    .orderBy("count", ascending=False)
)
 
display(state_counts)

## Compare the columns in both datasets

In [0]:
print("UFO Sightings columns:")
print(df_sightings_clean.columns)
 
print("\nUFO Truth columns:")
print(df_truth.columns)

## Join  the two datasets 
the two dartasets dp not share a city column so we will combine them using the shape olumn
this allows us tro merge ufo sightings with additional investogation information such as weather, witnesses and research outcomes

# Inner join = only rows where the UFO shape appears in both datasets

In [0]:
combined_df = df_sightings_clean.join(
    df_truth,
    on="shape",
    how="inner"
)

display(combined_df)

## Visualisation: UFO sightings by shape

Now that the dataset is combined, we can count how many sightings exist for each UFO shape.

In [0]:
shape_counts = (
    combined_df
    .groupBy("shape")
    .count()
    .orderBy("count", ascending=False)
)
 
display(shape_counts)

## visualisation: research Outcomes by UFO shape

The investigation dataset contains a column called **researchOutcome**. This shows what the investigators concluded about the sighting

In [0]:
outcome_by_shape = (
    combined_df
    .groupBy("shape", "researchOutcome")
    .count()
    .orderBy("count", ascending=False)
)
 
display(outcome_by_shape)

## Visualisation: Physical evidence reported 

This chart shows whether physical evidence was reported for each UFO shape.

In [0]:
evidence_by_shape = (
    combined_df
    .groupBy("shape", "physicalEvidence")
    .count()
    .orderBy("count", ascending=False)
)
 
display(evidence_by_shape)

## Visulisation:Weather conditions during sightings 

In [0]:
weather_by_shape = (
    combined_df
    .groupBy("shape", "weather")
    .count()
    .orderBy("count", ascending=False)
)
 
display(weather_by_shape)

## Visualisation 5: Average number of witnesses by UFO shape
This chart show which UFO shapes tend to have the highest amount of witnesses.

In [0]:
from pyspark.sql.functions import avg, col
from pyspark.sql.types import IntegerType
 
combined_witnesses = combined_df.withColumn(
    "witnesses_num",
    col("witnesses").cast(IntegerType())
)
 
avg_witnesses = (
    combined_witnesses
    .groupBy("shape")
    .agg(avg("witnesses_num").alias("avg_witnesses"))
    .orderBy("avg_witnesses", ascending=False)
)
 
display(avg_witnesses)

## Visualisation 7: Number of UFO sightings by state
This chart shows which states gave the most sightings.

In [0]:
state_counts = (
    df_sightings_clean
    .groupBy("state")
    .count()
    .orderBy("count", ascending=False)
)
 
display(state_counts)

## visualisation 8: Map of UFO Sightings 
this map uses longtitude and latitude to plot sightings geographically

## Prepare coordinates for mapping 

Longtitude and latitude must be numeric values for the map visualisations to work.

In [0]:
from pyspark.sql.functions import col
from pyspark.sql.types import DoubleType
 
map_df = (
    df_sightings_clean
    .withColumn("latitude", col("latitude").cast(DoubleType()))
    .withColumn("longitude", col("longitude").cast(DoubleType()))
)
 
display(map_df)

## Remove rows with missing coordinates

In [0]:
map_sample = map_df.select (
    "latitude",
    "longitude",
    "Shape",
    "country"
).limit(5000)
 
display(map_sample)

In [0]:
display(map_sample)

##We joined a UFO shape because both datasets contain that column.
##In real data projects, joins normally use a unique ID, but for this exercise we are demonstrating how Spark joins datasets.